#
**Version 2.1 — Effective Date: March 8, 2026**

This notebook serves as the technical implementation of my professional standards.
> **Core Philosophy:** I solve real-world business problems by applying rigorous engineering standards to messy data.

In [ ]:
# COMMANDMENT 0: INFRASTRUCTURE SETUP
import os
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets

# 1. THE FIX: Enable experimental features before importing IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer

# 2. UPDATED FOR REGRESSION: Moving from Classification to Gradient Prediction
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Model Selection: Swapped Logistic/RandomForestClassifier for Regressors
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor # Ensuring you have the tool for your XGBoost goal

# Metrics: Swapped Confusion Matrix for Regression Metrics (MSE, R2)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Infrastructure Setup Complete. Experimental Imputers enabled.")
print("🚀 Models updated for continuous target (Regression) mode.")

In [ ]:
# --- 1. LOAD LOCAL FILES ---
# Ensure 'product_master.csv' and 'clickstream_normal.csv' are in your current folder
try:
    df_p = pd.read_csv('product_master.csv')
    df_s = pd.read_csv('clickstream_normal.csv')
    print("✅ Files loaded successfully.")
except FileNotFoundError:
    print("❌ Error: CSV files not found. Please run your data creation cell first.")

# --- 2. FEATURE ENGINEERING (Friction Metrics) ---
# We aggregate the clickstream into SKU-level performance metrics
agg = df_s.groupby('sku_id').agg(
    views=('session_id', 'count'),
    carts=('action', lambda x: (x==1).sum()),
    buys=('action', lambda x: (x==2).sum())
).reset_index()

# Calculate Cart-to-Detail (CtD) and Buy-to-Detail (BtD) Ratios
agg['ctd_ratio'] = agg['carts'] / agg['views'].replace(0, 1)
agg['btd_ratio'] = agg['buys'] / agg['views'].replace(0, 1)

# Add event time for SageMaker Feature Store compatibility
agg['event_time'] = time.time()

# --- 3. MERGE & CONTINUOUS TARGET LOGIC ---
# Merge behavioral data with product attributes (inventory/cost)
final = pd.merge(df_p, agg, on='sku_id', how='left').fillna(0)

# Calculate the "Friction Score" (The gap between interest and actual sales)
# High friction = High CtD but low BtD
final['friction_score'] = (final['ctd_ratio'] - (final['btd_ratio'] * 2)).clip(0, 1)

# Normalize Inventory Pressure (Scaling 0 to 1)
# This ensures we only discount heavily if we have trapped capital
final['inv_pressure'] = final['inventory_level'] / final['inventory_level'].max()

# GENERATE THE GRADIENT TARGET (The "Elasticity Dial")
# Range: 0.0 (No Change) to -0.20 (20% Discount)
# This prevents the 'Binary Brain' issue seen in previous EDA
final['target_adjustment'] = -0.20 * (final['friction_score'] * final['inv_pressure'])
final['target_adjustment'] = final['target_adjustment'].round(3)

# --- 4. EXPORT FOR EXPERIMENTATION ---
final.to_csv('features.csv', index=False)

# --- 5. VERIFY GRADIENT ---
print(f"Total Records: {len(final)}")
print(f"Unique Target Values: {final['target_adjustment'].nunique()}")
print("\nTarget Distribution Preview:")
print(final['target_adjustment'].value_counts().head(20))

if final['target_adjustment'].nunique() > 2:
    print("\n✅ Gradient confirmed: The model now has a continuous scale to learn from.")
else:
    print("\n⚠️ Warning: Target is still too clumped. Check inventory variance.")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# COMMANDMENT 1: THE WALL OF SILENCE
# 1. Load the features.csv file directly
df = pd.read_csv('features.csv')

# 2. Split the data
# Directly splitting the full dataframe before any further analysis or column isolation
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42
)

print(f"✅ Data Load & Split Complete.")
print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")

In [ ]:
# COMMANDMENT 2: MISSINGNESS AS A SIGNAL
def add_missing_indicators(df):
    """
    Captures 'Invisibility' as a feature.
    If a ratio is missing, it's a signal that the SKU had 0 traffic.
    """
    df_copy = df.copy()
    # We only look for missingness in the feature columns
    cols_to_check = df_copy.columns

    for col in cols_to_check:
        null_count = df_copy[col].isnull().sum()
        if null_count > 0:
            # Create the indicator: 1 if missing, 0 if present
            df_copy[f'{col}_is_missing'] = df_copy[col].isnull().astype(int)

            # For pricing regression, we fill the actual NaN with 0
            # so the XGBoost model can still process the row
            df_copy[col] = df_copy[col].fillna(0)

    return df_copy

# Apply to your stratified splits
train_df = add_missing_indicators(train_df)
test_df = add_missing_indicators(test_df)

# Quick check: See if any 'invisible' products were flagged
missing_cols = [c for c in train_df.columns if '_is_missing' in c]
print(f"✅ Missing value signals captured: {missing_cols}")
print(f"Preview of missingness features:\n{train_df[missing_cols].head() if missing_cols else 'No missing values found in this sample.'}")

In [ ]:
# COMMANDMENT 3: THE TRANSFORMATION ENGINE
# 1. Isolate feature types from your processed train_df
# We exclude the target 'target_adjustment' and metadata like 'sku_id'
features_to_use = [col for col in train_df.columns if col not in ['target_adjustment', 'sku_id', 'product_name', 'event_time', 'friction_score']]

numeric_features = train_df[features_to_use].select_dtypes(include=['int64', 'float64']).columns
categorical_features = train_df[features_to_use].select_dtypes(include=['object']).columns

# 2. Define Transformers
# For numeric: Median imputation (safety) + Scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# For categorical: OneHotEncoding the 5 Categories (Audio, etc.)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Create the Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. Fit the preprocessor on TRAIN only to prevent leakage
# This prepares the engine to transform both train and test sets identically
preprocessor.fit(train_df[features_to_use])

print("✅ Pipeline Preprocessor defined and fitted.")
print(f"Features being scaled: {list(numeric_features)}")
print(f"Features being encoded: {list(categorical_features)}")

In [ ]:
# COMMANDMENT 4 & 5: BASELINE VS SOTA & KPI SELECTION
# We switch to Mean Absolute Error (MAE).
# This tells us, on average, how many percentage points off our price suggestion is.
SCORING_METRIC = 'neg_mean_absolute_error'

# 1. Simple Baseline (Glass Box) - Linear Regression
# Good for understanding the basic linear relationship between inventory and price.
baseline_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('m', LinearRegression())
])

# 2. Complex Model (SOTA) - XGBoost Regressor
# The "Heavy Lifter" that captures non-linear price elasticity.
complex_model = Pipeline(steps=[
    ('pre', preprocessor),
    ('m', XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42))
])

print(f"✅ Regression Models ready.")
print(f"Targeting: {SCORING_METRIC} (Lower MAE = More accurate price suggestions)")

In [ ]:
# COMMANDMENT 6: STABILITY CHECK
from sklearn.model_selection import KFold

def validate_stability(model, name, X, y):
    # Use KFold for continuous regression targets
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # cross_validate returns negative values for MAE
    scores = cross_validate(model, X, y, cv=cv, scoring=SCORING_METRIC)

    # We take the absolute to make it readable (e.g., 0.015 error)
    mean_mae = np.abs(scores['test_score'].mean())
    std_mae = scores['test_score'].std()

    print(f"🚀 {name} MAE: {mean_mae:.4f} (+/- {std_mae:.4f})")
    return mean_mae

# We use the raw dataframes here because the Pipeline handles the preprocessor
b_score = validate_stability(baseline_model, "Baseline", train_df, train_df['target_adjustment'])
c_score = validate_stability(complex_model, "Complex", train_df, train_df['target_adjustment'])

# Rule 4 Check: Did the Complex model reduce error by >5%?
# In regression, a LOWER score is better, so we check if Complex < Baseline
improvement = (b_score - c_score) / b_score
print(f"📈 Error Reduction: {improvement:.2%}")

if improvement > 0.05:
    print("💎 Decision: The Complex model is significantly more accurate. Proceed to Tuning.")
else:
    print("⚠️ Decision: Baseline is sufficient. The complexity of XGBoost isn't adding enough value yet.")

In [ ]:
# COMMANDMENT 7 & 8: ERROR ANALYSIS & RIGOR
# 1. Fit the model on the training data
complex_model.fit(train_df, train_df['target_adjustment'])

# 2. Predict on the test set
y_pred = complex_model.predict(test_df)
y_true = test_df['target_adjustment']

# 3. Numeric Metrics (The "Stats")
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"--- POST-MORTEM REPORT ---")
print(f"Mean Absolute Error (MAE): {mae:.5f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.5f}")
print(f"R² Score: {r2:.4f}")

# 4. Visual Analysis: Predicted vs. Actual
plt.figure(figsize=(12, 5))

# Plot 1: Prediction Error
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_true, y=y_pred, alpha=0.6)
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], '--r', lw=2)
plt.title("Actual vs. Predicted Adjustments")
plt.xlabel("True Discount")
plt.ylabel("Predicted Discount")

# Plot 2: Residual Distribution (Are we biased?)
plt.subplot(1, 2, 2)
residuals = y_true - y_pred
sns.histplot(residuals, kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title("Residual Distribution (Error)")
plt.xlabel("Error Magnitude")

plt.tight_layout()
os.makedirs('visuals', exist_ok=True)
plt.savefig('visuals/error_analysis.png')
plt.show()

In [ ]:
# COMMANDMENT 4 & 5: BASELINE VS SOTA & HYPERPARAMETER TUNING (300 ITEM SCALE)
SCORING_METRIC = 'neg_mean_absolute_error'

# 1. Complex Model Pipeline
xgb_pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('m', XGBRegressor(random_state=42, objective='reg:absoluteerror'))
])

# 2. Expanded Search Space for 300 Items
param_grid = {
    'm__n_estimators': [100, 200],         # We can afford more estimators now
    'm__max_depth': [3, 4, 5],             # Testing slightly more depth
    'm__learning_rate': [0.05, 0.1],       # Focused on higher stability
    'm__reg_lambda': [1, 5, 10]            # Testing various regularization levels
}

# 3. Running the Search
grid_search = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=3,
    scoring=SCORING_METRIC,
    verbose=1
)

print("🚀 Searching for the optimal 300-item pricing logic...")
grid_search.fit(train_df, train_df['target_adjustment'])

# Set the complex_model to the best discovered version
complex_model = grid_search.best_estimator_

print(f"✅ Best Parameters Found: {grid_search.best_params_}")
print(f"✅ Best CV Score (MAE): {abs(grid_search.best_score_):.5f}")

In [ ]:
from sklearn.model_selection import learning_curve

# COMMANDMENT 11: THE GROWTH MAP
train_sizes, train_scores, test_scores = learning_curve(
    complex_model, train_df, train_df['target_adjustment'],
    cv=3, scoring=SCORING_METRIC, train_sizes=np.linspace(0.1, 1.0, 5)
)

# Convert to positive MAE for plotting
train_mean = -np.mean(train_scores, axis=1)
test_mean = -np.mean(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label="Training Error")
plt.plot(train_sizes, test_mean, 's-', label="Cross-Validation Error")
plt.title("Is 100 Items Enough? (Learning Curve)")
plt.xlabel("Number of Training Samples")
plt.ylabel("MAE (Lower is Better)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# COMMANDMENT 7 & 8: FINAL RIGOR (POST-TUNING)
# 1. Ensure we are using the BEST model from your GridSearch
# If you just ran the GridSearch, complex_model is already updated.

# 2. Predict on the test set using the full DataFrame
# The pipeline handles selecting 'features_to_use' and transforming them
y_pred = complex_model.predict(test_df)
y_true = test_df['target_adjustment']

# 3. Numeric Metrics (The "Stats")
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"--- POST-TUNING POST-MORTEM ---")
print(f"Mean Absolute Error (MAE): {mae:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"R² Score: {r2:.4f}")

# 4. Visual Analysis
plt.figure(figsize=(12, 5))

# Plot 1: Prediction Error
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_true, y=y_pred, alpha=0.7, s=100, edgecolor='w')
# Draw the "Perfect Prediction" line
plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], '--r', lw=2, label='Perfect Fit')
plt.title("Actual vs. Predicted (Tuned)")
plt.xlabel("True Discount")
plt.ylabel("Model's Suggested Discount")
plt.legend()

# Plot 2: Residual Distribution
plt.subplot(1, 2, 2)
residuals = y_true - y_pred
sns.histplot(residuals, kde=True, color='teal', bins=15)
plt.axvline(0, color='red', linestyle='--')
plt.title("Residuals (Are we biased?)")
plt.xlabel("Error Magnitude")

plt.tight_layout()
plt.show()

In [ ]:
# COMMANDMENT 12: THE STRESS TEST (ANTI-LUCK PROTOCOL)
# 1. Create a "Dirty" copy of your test data
dirty_test = test_df.copy()

# 2. Inject 30% Random Noise into your key "Friction" signals
noise_factor = 0.30
dirty_test['ctd_ratio'] *= (1 + np.random.uniform(-noise_factor, noise_factor, size=len(dirty_test)))
dirty_test['inventory_level'] += np.random.randint(-5, 6, size=len(dirty_test))
dirty_test['inv_pressure'] *= (1 + np.random.uniform(-noise_factor, noise_factor, size=len(dirty_test)))

# 3. Predict on the Chaos
y_pred_dirty = complex_model.predict(dirty_test)
y_true_dirty = dirty_test['target_adjustment']

# 4. Compare Results
mae_dirty = mean_absolute_error(y_true_dirty, y_pred_dirty)
r2_dirty = r2_score(y_true_dirty, y_pred_dirty)

print(f"--- STRESS TEST RESULTS ---")
print(f"Original R²: {r2:.4f}")
print(f"Dirty Data R²: {r2_dirty:.4f}")
print(f"Accuracy Retention: {(r2_dirty/r2):.2%}")



In [ ]:
# 1. Extract feature names from the preprocessor
# This gives us names like 'num__inventory_level' and 'cat__category_Audio'
feature_names = complex_model.named_steps['pre'].get_feature_names_out()

# 2. Get the importance scores from the XGBoost model step ('m')
importances = complex_model.named_steps['m'].feature_importances_

# 3. Create a sorted series for plotting
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

# 4. Visualize the Drivers
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='magma')

plt.title('Pricing Engine: Key Drivers of Discount Suggestions', fontsize=14)
plt.xlabel('Importance Score (Information Gain)', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

print(f"✅ Top Price Driver: {feat_imp.index[0]}")
print(f"✅ Second Driver: {feat_imp.index[1]}")

In [ ]:
# COMMANDMENT 10: VALIDATION LAYER (STAKEHOLDER DASHBOARD)
def quick_predict(**kwargs):
    # 1. Convert input to DataFrame
    row = pd.DataFrame([kwargs])

    # 2. Add any missing columns that the model expects (like categories)
    # We default them to 0 (False) for the one-hot encoded columns
    for col in train_df.columns:
        if col not in row.columns and col != 'target_adjustment':
            row[col] = 0

    # 3. Generate the suggested discount
    suggested_adj = complex_model.predict(row)[0]

    # 4. Logic: Calculate the Impact
    current_msrp = kwargs.get('current_msrp', 0)
    new_price = current_msrp * (1 + suggested_adj)

    print(f"\n--- PRICING ENGINE SUGGESTION ---")
    print(f"Strategic Adjustment: {suggested_adj*100:.2f}%")
    print(f"New Recommended Price: ${new_price:.2f}")
    print(f"Status: {'🟢 PROTECT MARGIN' if suggested_adj > -0.05 else '🟡 CLEAR INVENTORY' if suggested_adj > -0.15 else '🔴 AGGRESSIVE CLEARANCE'}")

# Setup sliders for the "Friction" and "Inventory" drivers
# Using the top drivers identified in your Feature Importance cell
ui_elements = {
    'current_msrp': widgets.FloatSlider(min=10, max=1000, step=10, value=250, description='MSRP ($)'),
    'inventory_level': widgets.IntSlider(min=0, max=500, value=100, description='Inventory'),
    'inv_pressure': widgets.FloatSlider(min=0, max=1, step=0.01, value=0.2, description='Inv Pressure'),
    'ctd_ratio': widgets.FloatSlider(min=0, max=0.5, step=0.01, value=0.05, description='Cart Ratio'),
    'btd_ratio': widgets.FloatSlider(min=0, max=0.2, step=0.01, value=0.02, description='Buy Ratio')
}

print("🎮 INTERACTIVE PRICING DASHBOARD")
widgets.interact(quick_predict, **ui_elements);